# E3 · The same pruning prunes nothing: `ccp_alpha` depends on the units of the target

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 40 minutes · **extended path**

## Objective

Find out why the same `ccp_alpha=5e-4` that gave 182 leaves on prices in hundreds of thousands of dollars leaves **2453 leaves** when the prices are in dollars: cost-complexity pruning uses the impurity (the squared error) in units of the target, so `alpha` has to be rescaled with the variance of `y`.

## Dataset

`ds.california()`: `y` in hundreds of thousands of dollars (`y_tr`, `y_te`) and the same in dollars (`y_tr_dollars = y_tr * 100000`, `y_te_dollars`). Tree: `DecisionTreeRegressor(min_samples_leaf=5, random_state=0, ccp_alpha=...)`, 75/25 split.

## Educational purpose

Recognize a parameter that depends on the **units** of the problem: `ccp_alpha` (regression) and `min_impurity_decrease` are expressed in impurity units, that is in the square of the target. A value found "on scale" is not transferable: change the units and the regularization disappears (or becomes excessive).

## Task

1. Run the colleague's cell: `ref_leaves` (pruning with `ccp_alpha=5e-4` on `y` in hundreds of thousands) and `bug_leaves` (same `ccp_alpha` on `y` in dollars), with the respective `ref_rmse` and `bug_rmse` (in **hundreds of thousands of dollars**, to compare them).
2. Save `scale_factor = 100000 ** 2` and `alpha_dollars = 5e-4 * scale_factor`: the equivalent `alpha` for the target in dollars.
3. Save `ok_leaves` and `ok_rmse` training the tree on the target in dollars with `alpha_dollars` (RMSE reported in hundreds of thousands). They must match `ref_leaves` and `ref_rmse`.
4. Write `relative_alpha(alpha, y)` that returns `alpha / np.var(y)`: a **dimensionless** parameter. Check that `relative_alpha(5e-4, y_tr)` and `relative_alpha(alpha_dollars, y_tr_dollars)` match and save the value in `rel_alpha`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
y_tr_dollars, y_te_dollars = y_tr * 100000, y_te * 100000


def tree(alpha, y_train):
    return DecisionTreeRegressor(min_samples_leaf=5, random_state=0, ccp_alpha=alpha).fit(X_tr, y_train)

## Your work

### The colleague's code

In [ ]:
t_ref = tree(5e-4, y_tr)                                         # target in hundreds of thousands of dollars
t_bug = tree(5e-4, y_tr_dollars)                                  # same alpha, target in dollars
ref_leaves, bug_leaves = t_ref.get_n_leaves(), t_bug.get_n_leaves()
ref_rmse = mean_squared_error(y_te, t_ref.predict(X_te), squared=False)
bug_rmse = mean_squared_error(y_te_dollars, t_bug.predict(X_te), squared=False) / 100000
print(f"leaves: {ref_leaves} (target in hundreds of thousands) against {bug_leaves} (in dollars) | RMSE: {ref_rmse:.3f} against {bug_rmse:.3f}")

### Your diagnosis and the fix

In [ ]:
scale_factor = None
alpha_dollars = None

In [ ]:
ok_leaves = None
ok_rmse = None

In [ ]:
def relative_alpha(alpha, y):
    ...


rel_alpha = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`alpha_dollars = 5e-4 * 100000 ** 2`; `tree(alpha_dollars, y_tr_dollars)`; RMSE: `mean_squared_error(y_te_dollars, m.predict(X_te), squared=False) / 100000`. `relative_alpha`: `return alpha / np.var(y)`.'):
    check(ref_leaves < 250 and bug_leaves > 2000 and bug_rmse > ref_rmse + 0.005, "the colleague: 182 leaves against over 2000 (no pruning), with a worse RMSE", "don't modify the colleague's code")
    check(scale_factor == 1e10 and close(alpha_dollars, 5e6), "the scale factor is 1e10 (the square of 100000); equivalent alpha 5e6", "impurity = squared error: it scales like the square of the target")
    check(ok_leaves == ref_leaves and abs(ok_rmse - ref_rmse) < 1e-6, "with the rescaled alpha the number of leaves and the error match those on the original scale", "tree(alpha_dollars, y_tr_dollars)")
    check(close(relative_alpha(5e-4, y_tr), rel_alpha) and close(relative_alpha(alpha_dollars, y_tr_dollars), rel_alpha), "relative alpha (alpha / variance of the target) equal in the two units", "alpha / np.var(y)")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_ccp_alpha_depends_on_the_target_scale.ipynb`. Open it after you pass the check, or after you have really tried.